In [ ]:

!pip install -U openai-whisper



In [ ]:
!sudo apt update && sudo apt install ffmpeg

In [11]:
import whisper
import os
import difflib
import string

# ===================== 1. 路径配置（适配魔搭持久化目录） =====================
BASE_DIR = "/mnt/workspace/speech/"
ORIGINAL_TXT = os.path.join(BASE_DIR, "original.txt")  # 朗读原文文件
AUDIO_DIR = os.path.join(BASE_DIR, "audio")            # 学生录音目录
OUTPUT_DIR = os.path.join(BASE_DIR, "output")          # 报告输出目录

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# 自动创建输出目录（不存在则新建，存在不报错）
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===================== 2. 工具函数定义 =====================
def preprocess_text(text):
    """文本标准化：小写、去标点、分词"""
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    word_list = [word.strip() for word in text.split() if word.strip()]
    return word_list

def calculate_fluency(word_list):
    """基于单词时间戳计算流利度指标"""
    if not word_list:
        return {}
    
    total_word_num = len(word_list)
    first_start = word_list[0]["start"]
    last_end = word_list[-1]["end"]
    total_duration = last_end - first_start
    
    total_speech_time = sum(w["duration"] for w in word_list)
    total_pause_time = total_duration - total_speech_time
    
    # 统计停顿：相邻单词间隔 > 0.3秒 记为一次有效停顿
    pauses = []
    for i in range(1, len(word_list)):
        gap = word_list[i]["start"] - word_list[i-1]["end"]
        if gap > 0.3:
            pauses.append(gap)
    
    pause_count = len(pauses)
    avg_pause = sum(pauses) / pause_count if pause_count > 0 else 0
    long_pause_num = sum(1 for p in pauses if p > 1.0)
    
    wpm = (total_word_num / total_duration) * 60 if total_duration > 0 else 0
    speech_ratio = (total_speech_time / total_duration) * 100 if total_duration > 0 else 0
    avg_word_dur = total_speech_time / total_word_num if total_word_num > 0 else 0
    
    return {
        "总朗读时长(秒)": round(total_duration, 2),
        "有效发音时长(秒)": round(total_speech_time, 2),
        "总停顿时长(秒)": round(total_pause_time, 2),
        "停顿次数": pause_count,
        "平均停顿时长(秒)": round(avg_pause, 2),
        "长停顿(>1秒)次数": long_pause_num,
        "语速(WPM)": round(wpm, 1),
        "语音占比(%)": round(speech_ratio, 1),
        "平均单词发音时长(秒)": round(avg_word_dur, 3)
    }

def fluency_grade(fluency_data):
    """根据流利度指标给出评价与得分"""
    wpm = fluency_data["语速(WPM)"]
    speech_ratio = fluency_data["语音占比(%)"]
    long_pause = fluency_data["长停顿(>1秒)次数"]
    
    feedback = []
    score = 0
    
    if wpm >= 100:
        feedback.append("✅ 语速较快，朗读节奏流畅")
        score += 35
    elif wpm >= 70:
        feedback.append("✅ 语速适中，节奏平稳自然")
        score += 30
    elif wpm >= 50:
        feedback.append("⚠️ 语速偏慢，存在较多思考间隙")
        score += 20
    else:
        feedback.append("❌ 语速过慢，连贯性不足")
        score += 10
    
    if speech_ratio >= 70:
        feedback.append("✅ 有效发音占比高，停顿很少")
        score += 35
    elif speech_ratio >= 55:
        feedback.append("✅ 发音占比正常，停顿合理")
        score += 30
    elif speech_ratio >= 40:
        feedback.append("⚠️ 停顿偏多，流畅度有待提升")
        score += 20
    else:
        feedback.append("❌ 停顿过多，朗读断断续续")
        score += 10
    
    if long_pause == 0:
        feedback.append("✅ 无明显长停顿，整体连贯")
        score += 30
    elif long_pause <= 2:
        feedback.append(f"⚠️ 存在{long_pause}处长停顿，对流畅度影响不大")
        score += 20
    elif long_pause <= 5:
        feedback.append(f"❌ 存在{long_pause}处长停顿，流畅度受明显影响")
        score += 10
    else:
        feedback.append(f"❌ 长停顿多达{long_pause}处，流畅度较差")
        score += 5
    
    if score >= 85:
        level = "优秀"
    elif score >= 70:
        level = "良好"
    elif score >= 50:
        level = "一般"
    else:
        level = "待加强"
    
    return {"流利度得分": score, "流利度等级": level, "详细评价": feedback}

def generate_report(student_name, recognized_text, accuracy, error_details, fluency_data, fluency_result, total_score, total_comment):
    """生成单学生的完整报告文本"""
    report = f"{'='*60}\n"
    report += f"学生朗读评估报告\n"
    report += f"学生姓名：{student_name}\n"
    report += f"{'='*60}\n\n"
    
    report += "【一、Whisper 识别全文】\n"
    report += recognized_text + "\n\n"
    
    report += "【二、发音准确度评估】\n"
    report += f"原文总单词数：{len(ref_words)}\n"
    report += f"正确匹配单词数：{int(accuracy/100 * len(ref_words))}\n"
    report += f"发音匹配准确率：{accuracy:.1f}%\n\n"
    report += "错误明细：\n"
    if error_details:
        for err in error_details:
            report += f"  {err}\n"
    else:
        report += "  ✅ 全部匹配，未发现发音错误\n"
    report += "\n"
    
    report += "【三、流利度指标详情】\n"
    for key, value in fluency_data.items():
        report += f"  {key}：{value}\n"
    report += "\n"
    
    report += "【四、流利度评价】\n"
    report += f"  流利度得分：{fluency_result['流利度得分']}/100\n"
    report += f"  流利度等级：{fluency_result['流利度等级']}\n"
    report += "  详细说明：\n"
    for item in fluency_result["详细评价"]:
        report += f"    {item}\n"
    report += "\n"
    
    report += "【五、综合总评】\n"
    report += f"  综合得分：{total_score:.1f}/100（准确度占90%，流利度占10%）\n"
    report += f"  {total_comment}\n"
    report += f"\n{'='*60}\n"
    
    return report

# ===================== 3. 预加载模型 + 读取原文（只执行一次） =====================
print("🔄 正在加载 Whisper 模型...")
# 如需更高精度，可将 "tiny" 替换为 "base" / "small"
model = whisper.load_model("turbo")
print("✅ 模型加载完成")

# 读取原文并标准化
with open(ORIGINAL_TXT, "r", encoding="utf-8") as f:
    original_text = f.read()
ref_words = preprocess_text(original_text)
print(f"✅ 原文读取完成，共 {len(ref_words)} 个单词")

# ===================== 4. 批量处理所有学生录音 =====================
# 获取audio目录下所有MP3文件
audio_files = [f for f in os.listdir(AUDIO_DIR) if f.lower().endswith(('.mp3','.m4a'))]
print(f"\n📂 共找到 {len(audio_files)} 个学生录音文件，开始批量处理...\n")

for idx, audio_file in enumerate(audio_files, 1):
    student_name = os.path.splitext(audio_file)[0]  # 提取文件名作为学生姓名
    audio_path = os.path.join(AUDIO_DIR, audio_file)
    output_path = os.path.join(OUTPUT_DIR, f"{student_name}.txt")
    
    print(f"[{idx}/{len(audio_files)}] 正在处理：{student_name}")
    
    try:
        # 4.1 Whisper 转写（带单词时间戳）
        result = model.transcribe(
            audio_path,
            temperature=0,
            word_timestamps=True
        )
        recognized_text = result["text"]
        
        # 提取带时间的单词列表
        recognized_words = []
        for segment in result["segments"]:
            if "words" in segment:
                for word_info in segment["words"]:
                    clean_word = word_info["word"].strip().lower().strip(string.punctuation)
                    if clean_word:
                        recognized_words.append({
                            "word": clean_word,
                            "start": word_info["start"],
                            "end": word_info["end"],
                            "duration": word_info["end"] - word_info["start"]
                        })
        
        # 4.2 发音准确度计算
        hyp_words = [w["word"] for w in recognized_words]
        matcher = difflib.SequenceMatcher(None, ref_words, hyp_words)
        opcodes = matcher.get_opcodes()
        
        correct_count = 0
        error_details = []
        for tag, i1, i2, j1, j2 in opcodes:
            if tag == 'equal':
                correct_count += (i2 - i1)
            elif tag == 'replace':
                original = ' '.join(ref_words[i1:i2])
                recognized = ' '.join(hyp_words[j1:j2])
                error_details.append(f"❌ 错读：原文「{original}」 → 识别为「{recognized}」")
            elif tag == 'delete':
                missed = ' '.join(ref_words[i1:i2])
                error_details.append(f"⚠️ 漏读：原文单词「{missed}」未朗读")
            elif tag == 'insert':
                extra = ' '.join(hyp_words[j1:j2])
                error_details.append(f"➕ 多读：额外读出「{extra}」")
        
        accuracy = (correct_count / len(ref_words)) * 100 if ref_words else 0
        
        # 4.3 流利度计算
        fluency_data = calculate_fluency(recognized_words)
        fluency_result = fluency_grade(fluency_data)
        
        # 4.4 综合评分
        total_score = accuracy * 0.9 + fluency_result["流利度得分"] * 0.1
        if total_score >= 90:
            total_comment = "整体表现优秀，发音准确且朗读流畅，继续保持。"
        elif total_score >= 75:
            total_comment = "整体表现良好，针对少量错误和停顿加强练习即可明显提升。"
        elif total_score >= 60:
            total_comment = "整体表现一般，建议先纠正易错单词，再逐步提升朗读流畅度。"
        else:
            total_comment = "整体有待加强，建议从单词发音基础开始练习，熟练后再提升语速。"
        
        # 4.5 生成报告并写入文件
        report_content = generate_report(
            student_name, recognized_text, accuracy, error_details,
            fluency_data, fluency_result, total_score, total_comment
        )
        
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(report_content)
        
        print(f"  ✅ 处理完成，报告已保存：{student_name}.txt")
    
    except Exception as e:
        print(f"  ❌ 处理失败：{str(e)}")
        continue

print(f"\n🎉 批量处理全部完成！所有报告已保存至：{OUTPUT_DIR}")


🔄 正在加载 Whisper 模型...
✅ 模型加载完成
✅ 原文读取完成，共 139 个单词

📂 共找到 3 个学生录音文件，开始批量处理...

[1/3] 正在处理：student1


/mnt/workspace/.venv/lib/python3.10/site-packages/triton/backends/nvidia/driver.c:9:10: fatal error: Python.h: 没有那个文件或目录
    9 | #include <Python.h>
      |          ^~~~~~~~~~
compilation terminated.
/mnt/workspace/.venv/lib/python3.10/site-packages/whisper/timing.py:42: UserWarning: Failed to launch Triton kernels, likely due to missing CUDA toolkit; falling back to a slower median kernel implementation...
  warnings.warn(
/mnt/workspace/.venv/lib/python3.10/site-packages/triton/backends/nvidia/driver.c:9:10: fatal error: Python.h: 没有那个文件或目录
    9 | #include <Python.h>
      |          ^~~~~~~~~~
compilation terminated.
/mnt/workspace/.venv/lib/python3.10/site-packages/whisper/timing.py:146: UserWarning: Failed to launch Triton kernels, likely due to missing CUDA toolkit; falling back to a slower DTW implementation...
  warnings.warn(
/mnt/workspace/.venv/lib/python3.10/site-packages/triton/backends/nvidia/driver.c:9:10: fatal error: Python.h: 没有那个文件或目录
    9 | #include <Python.h>
  

  ✅ 处理完成，报告已保存：student1.txt
[2/3] 正在处理：student2


/mnt/workspace/.venv/lib/python3.10/site-packages/triton/backends/nvidia/driver.c:9:10: fatal error: Python.h: 没有那个文件或目录
    9 | #include <Python.h>
      |          ^~~~~~~~~~
compilation terminated.
/mnt/workspace/.venv/lib/python3.10/site-packages/whisper/timing.py:42: UserWarning: Failed to launch Triton kernels, likely due to missing CUDA toolkit; falling back to a slower median kernel implementation...
  warnings.warn(
/mnt/workspace/.venv/lib/python3.10/site-packages/triton/backends/nvidia/driver.c:9:10: fatal error: Python.h: 没有那个文件或目录
    9 | #include <Python.h>
      |          ^~~~~~~~~~
compilation terminated.
/mnt/workspace/.venv/lib/python3.10/site-packages/whisper/timing.py:146: UserWarning: Failed to launch Triton kernels, likely due to missing CUDA toolkit; falling back to a slower DTW implementation...
  warnings.warn(
/mnt/workspace/.venv/lib/python3.10/site-packages/triton/backends/nvidia/driver.c:9:10: fatal error: Python.h: 没有那个文件或目录
    9 | #include <Python.h>
  

  ✅ 处理完成，报告已保存：student2.txt
[3/3] 正在处理：student3


/mnt/workspace/.venv/lib/python3.10/site-packages/triton/backends/nvidia/driver.c:9:10: fatal error: Python.h: 没有那个文件或目录
    9 | #include <Python.h>
      |          ^~~~~~~~~~
compilation terminated.
/mnt/workspace/.venv/lib/python3.10/site-packages/whisper/timing.py:42: UserWarning: Failed to launch Triton kernels, likely due to missing CUDA toolkit; falling back to a slower median kernel implementation...
  warnings.warn(
/mnt/workspace/.venv/lib/python3.10/site-packages/triton/backends/nvidia/driver.c:9:10: fatal error: Python.h: 没有那个文件或目录
    9 | #include <Python.h>
      |          ^~~~~~~~~~
compilation terminated.
/mnt/workspace/.venv/lib/python3.10/site-packages/whisper/timing.py:146: UserWarning: Failed to launch Triton kernels, likely due to missing CUDA toolkit; falling back to a slower DTW implementation...
  warnings.warn(
/mnt/workspace/.venv/lib/python3.10/site-packages/triton/backends/nvidia/driver.c:9:10: fatal error: Python.h: 没有那个文件或目录
    9 | #include <Python.h>
  

  ✅ 处理完成，报告已保存：student3.txt

🎉 批量处理全部完成！所有报告已保存至：/mnt/workspace/speech/output


/mnt/workspace/.venv/lib/python3.10/site-packages/triton/backends/nvidia/driver.c:9:10: fatal error: Python.h: 没有那个文件或目录
    9 | #include <Python.h>
      |          ^~~~~~~~~~
compilation terminated.
/mnt/workspace/.venv/lib/python3.10/site-packages/triton/backends/nvidia/driver.c:9:10: fatal error: Python.h: 没有那个文件或目录
    9 | #include <Python.h>
      |          ^~~~~~~~~~
compilation terminated.
